<h1 align="center">PySpark离线开发实例(脱敏演示)</h1>

#### 一、导入PySpark核心模块

In [ ]:
from pyspark import SparkContext
from pyspark.sql import SparkSession, HiveContext, SQLContext
from pyspark.sql.functions import regexp_replace, trim, when, length
from pyspark.sql.types import StringType, NullType
import os
import sys

#### 二、解决Python2/3兼容性问题，设置编码为UTF-8

In [ ]:
reload(sys)
sys.setdefaultencoding('utf8')

#### 三、定义业务相关参数配置

##### Hudi表配置

In [ ]:
database = '***'                                                       # 目标库名
table_name = '***'                                                     # 目标表名
table_path = '***'.format(database, table_name)                        # 目标表在HDFS上的存储路径（按库名+表名拼接）
primaryKey = '***'                                                     # Hudi表的主键（必须唯一，用于upsert更新）
precombine_field = '***'                                               # 合并字段（更新时，用这个字段判断谁是最新数据）
partition_path_field = '***'                                           # 分区字段（按这个字段做分区，提升查询性能）
primaryKeyFiled = ""                                                   # 复合主键，单主键可以不填

##### Hive连接配置

In [ ]:
hivejdbcurl = '***'                     # Hive JDBC连接地址 
hivejdbcusername = "***"                # Hive用户名
hivejdbcpassword = "***"                # Hive密码

#### 四、定义数据生成/读取的SQL

 这里放一个我工作中最常见的异构表合并的例子。

In [ ]:
SQL = """
      --describe dwd.*** 
      --describe dwd.***
      --describe dwd.***

      --describe ads.***
      
      --select * from ***
      
      with 
      t1 as 
      (
        select 
                 a1 as A
                 b1 as B
                 c1 as C
               null as D
               null as E
        from(
              select a1,b1,c1,rksj,
                     row_number()over(partition by 用主键字段去重 order by 入库时间 desc) as rn
              from dwd.***
            ) p
        where p.rn=1
      ),

      t2 as 
      (
        select 
                 a2 as A
                 b2 as B
                 c2 as C
                 d2 as D
               null as E
        from(
              select a2,b2,c2,d2,rksj,
                     row_number()over(partition by 用主键字段去重 order by 入库时间 desc) as rn
              from dwd.***
            ) p
        where p.rn=1
      ),

      t3 as 
      (
        select 
                 a3 as A
               null as B
                 c3 as C
                 d3 as D
               null as E
        from(
              select a3,c3,d3,rksj,dt,
                     row_number()over(partition by 用主键字段去重 order by 入库时间 desc) as rn
              from dwd.***
            ) p
        where p.rn=1 and dt=(select max(dt) from dwd.***)
      )

      select * from(
                          select * from t1
                     union
                          select * from t2
                     union    
                          select * from t3
                   )
      """.format(primaryKeyFiled)

#### 五、初始化SparkSession

##### 配置Spark集群参数、开启Hive支持

In [ ]:
if __name__ == '__main__':
    spark = SparkSession \
            .builder \
            # 配置Spark动态分区写入（Hive数仓写入必备）
            .config("spark.hadoop.hive.exec.dynamic.partition", True) \     
            .config("spark.hadoop.hive.exec.dynamic.partition.mode", "nonstrict") \
            # 广播小表阈值（join时自动广播小表，优化性能）
            .config("spark.sql.autoBroadcastJoinThreshold", "100000") \
            # 序列化方式优化，提升性能、减少内存占用
            .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
            # 开启Hive支持，让Spark能读写Hive表
            .enableHiveSupport() \
            .getOrCreate()

##### 设置日志级别为ERROR，减少控制台无用日志输出

In [ ]:
spark.sparkContext.setLogLevel('ERROR')

#### 六、数据标准化

##### 执行SQL，得到DataFrame数据

In [ ]:
cleaned_table = spark.sql(SQL)

##### 去除所有字段的首尾空格

In [ ]:
 cleaned_table = cleaned_table.select([trim(cleaned_table[col]).alias(col) for col in cleaned_table.columns])

##### 替换所有字段中的换行符、制表符（避免写入Hudi/Hive时出错）

In [ ]:
 cleaned_table = cleaned_table.select([
        when(length(cleaned_table[col]) == 0, None)
        .otherwise(regexp_replace(cleaned_table[col], r'[\n\t\r]', ''))
        .alias(col) for col in cleaned_table.columns
    ])

##### 把所有NullType字段转为StringType（统一类型，避免写入报错）

In [ ]:
scm = cleaned_table.schema
    for x in scm:
        if x.dataType == NullType():
            x.dataType = StringType()
    cleaned_table = spark.createDataFrame(cleaned_table.rdd, scm)

#### 七、配置Hudi、写入数仓

#### Hudi写入配置

In [ ]:
 hudi_options = {
        'hoodie.table.name': table_name,                                      # 目标表名
        'hoodie.datasource.write.operation': 'upsert',                        # 写入模式：增量更新（存在则更新，不存在则插入）
        'hoodie.datasource.write.recordkey.field': primaryKey,                # 主键字段
        'hoodie.datasource.write.precombine.field': precombine_field,         # 合并字段
        'hoodie.datasource.write.partitionpath.field': partition_path_field,  # 分区字段
        'hoodie.datasource.write.keygenerator.class': 'org.apache.hudi.keygen.NonpartitionedKeyGenerator',
        'hoodie.datasource.write.table.name': table_name
    }

#### Hive同步配置：把Hudi表的元数据同步到Hive，方便后续查询

In [ ]:
hive_sync_options = {
        'hoodie.datasource.hive_sync.jdbcurl': hivejdbcurl,
        'hoodie.datasource.hive_sync.username': hivejdbcusername,
        'hoodie.datasource.hive_sync.password': hivejdbcpassword,
        'hoodie.datasource.hive_sync.enable': 'true',
        'hoodie.datasource.hive_sync.database': database,
        'hoodie.datasource.hive_sync.table': table_name,
        'hoodie.datasource.hive_sync.partition_fields': partition_path_field,
        'hoodie.datasource.hive_sync.partition_extractor_class': 'org.apache.hudi.hive.NonPartitionedExtractor'
    }

#### 合并所有配置

In [ ]:
hudi_options.update(hive_sync_options)

#### 执行写入：把清洗好的DataFrame写入Hudi表

In [ ]:
cleaned_table.write.format('hudi').options(**hudi_options).mode('overwrite').save(table_path)

#### 结束~

In [ ]:
print('保存到ADS层成功')